In [1]:
import pickle

with open("X_train_tfidf.pkl", "rb") as f:
    X_train = pickle.load(f)

with open("X_test_tfidf.pkl", "rb") as f:
    X_test = pickle.load(f)

with open("y_train_tfidf.pkl", "rb") as f:
    y_train = pickle.load(f)

with open("y_test_tfidf.pkl", "rb") as f:
    y_test = pickle.load(f)

with open("tfidf.pkl", "rb") as f:
    tfidf = pickle.load(f)

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'using device : {device}')

using device : cuda


In [4]:
import numpy as np

In [5]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.from_numpy(features.toarray()).float() #Converts sparse matrix → dense NumPy array
        self.labels = torch.from_numpy(labels.to_numpy()).float() #Converts Pandas Series → NumPy array

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [6]:
train_ds = CustomDataset(X_train, y_train)
test_ds = CustomDataset(X_test, y_test)

In [7]:
train_loader = DataLoader(train_ds, batch_size = 32, shuffle = True, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size = 32, shuffle = False, pin_memory=True)

In [8]:
class MyNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features,32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(p = 0.3),
            nn.Linear(32,16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(p = 0.3),
            nn.Linear(16,1)
        )
    def forward(self, x):
        return self.model(x)

In [29]:
epochs = 5
learning_rate = 0.001

In [30]:
model = MyNN(X_train.shape[1])
model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr = learning_rate)

In [31]:
for epoch in range(epochs):

    total_epoch_loss = 0

    for batch_features, batch_labels in train_loader:

        batch_features = batch_features.to(device)

        batch_labels = batch_labels.unsqueeze(1).float().to(device)

        outputs = model(batch_features)

        loss = criterion(outputs, batch_labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_epoch_loss += loss.item()
    avg_loss = total_epoch_loss/len(train_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss}")

Epoch 1, Loss: 0.6677788649025548
Epoch 2, Loss: 0.6084474802473953
Epoch 3, Loss: 0.5856189176268961
Epoch 4, Loss: 0.5604579103860818
Epoch 5, Loss: 0.532829017017993


In [32]:
model.eval()

total = 0
correct = 0

with torch.no_grad():

    for batch_features, batch_labels in test_loader:

        batch_features = batch_features.to(device)

        batch_labels = batch_labels.unsqueeze(1).float().to(device)

        outputs = model(batch_features)

        probabilities = torch.sigmoid(outputs)

        predicted = (probabilities >= 0.5).float()

        total += batch_labels.size(0)

        correct += (predicted == batch_labels).sum().item()

print(correct / total)

0.7491590581451225


In [33]:
model.eval()

total = 0
correct = 0

with torch.no_grad():

    for batch_features, batch_labels in train_loader:

        batch_features = batch_features.to(device)

        batch_labels = batch_labels.unsqueeze(1).float().to(device)

        outputs = model(batch_features)

        probabilities = torch.sigmoid(outputs)

        predicted = (probabilities >= 0.5).float()

        total += batch_labels.size(0)

        correct += (predicted == batch_labels).sum().item()

print(correct / total)

0.8601465817613841
